In [42]:
import os
import json
import re

# Folder path where the JSON files are stored
folder_path = "../json"

In [43]:
# Function to process the subtype field
def process_subtype(subtype_value):
    """Ensure the subtype field is a list."""
    if "/" in subtype_value:
        # Split the string by "/" and create a list
        return [entry.strip() for entry in subtype_value.split("/")]
    # If no "/", return the original value as a list
    return [subtype_value.strip()]

In [44]:
def modify_keywords(keywords):
    """Modify the 'Armor' keyword in the keywords list."""
    modified_keywords = []

    for keyword in keywords:
        keyword_name = keyword.get("keyword")
        
        # Only modify the "Armor" keyword
        if keyword_name == "Armor":
            target_value = keyword.get("target")
            if target_value:
                # If target_value contains colors, split them into a list
                modified_keyword = {
                    "keyword_name": "Armor",
                    "modifier_type": "COLOR",
                    "modifier_value": [color.strip() for color in target_value.split(",")]
                }
                modified_keywords.append(modified_keyword)
            else:
                modified_keywords.append(keyword)  # Keep the original if no target
        else:
            modified_keywords.append(keyword)  # Keep non-"Armor" keywords as they are

    return modified_keywords

In [45]:
def update_keywords_structure(keywords):
    """
    Update the 'keywords' structure by renaming 'modifier_value' to 'keyword_modifier' 
    and deleting 'modifier_type' if the structure matches.
    """
    updated_keywords = []

    for keyword in keywords:
        # Check if the structure contains 'keyword_name', 'modifier_type', and 'modifier_value'
        if (
            "keyword_name" in keyword and
            "modifier_type" in keyword and
            "modifier_value" in keyword
        ):
            updated_keywords.append({
                "keyword_name": keyword["keyword_name"],
                "keyword_modifier": keyword["modifier_value"]  # Rename 'modifier_value' to 'keyword_modifier'
            })
        else:
            # Keep the keyword as is if the structure doesn't match
            updated_keywords.append(keyword)

    return updated_keywords

In [46]:
def update_crush_effect(effect):
    """
    Update a crush effect:
    - Extract the integer from 'target' and assign it to 'keyword_modifier'.
    - Update 'keywords' structure.
    """
    if "keywords" in effect:
        for keyword in effect["keywords"]:
            if keyword.get("keyword") == "Crush":
                # Extract the integer from 'target' (e.g., "+5")
                if "target" in keyword and keyword["target"].startswith("+"):
                    try:
                        keyword_modifier = int(keyword["target"].lstrip("+"))
                        keyword["keyword_modifier"] = keyword_modifier
                    except ValueError:
                        print(f"Invalid target format in effect: {effect}")
                        keyword["keyword_modifier"] = None
                else:
                    keyword["keyword_modifier"] = None

                # Update 'keywords' structure
                keyword["keyword_name"] = keyword.pop("keyword")
                
                # Remove unused fields
                keyword.pop("target", None)
                keyword.pop("value", None)

    return effect

In [47]:
def update_burst_effect(effect):
    """
    Update a burst effect:
    - Move 'target' value to 'condition'.
    - Update 'keywords' structure.
    """
    if "keywords" in effect:
        for keyword in effect["keywords"]:
            if keyword.get("keyword") == "Burst":
                # Move 'target' to 'condition'
                if "target" in keyword:
                    effect["condition"] = keyword["target"]

                # Update 'keywords' structure
                keyword["keyword_name"] = keyword.pop("keyword")
                keyword["keyword_modifier"] = keyword.pop("value", None)

                # Remove 'target'
                keyword.pop("target", None)

    return effect

In [48]:
def update_gale_effect(effect):
    """Update the structure for the 'Gale' keyword."""
    if "keywords" in effect:
        for keyword in effect["keywords"]:
            if keyword.get("keyword") == "Gale":
                # Update to the new structure
                effect["keywords"] = [
                    {
                        "keyword_name": keyword["keyword"],
                        "keyword_modifier": int(keyword["target"])  # Convert target to an integer
                    }
                ]
    return effect

In [49]:
def update_immortal_effect(effect):
    """Update the structure for the 'Immortal' keyword."""
    if "keywords" in effect:
        for keyword in effect["keywords"]:
            if keyword.get("keyword") == "Immortal":
                # Extract all integers from the target field
                target = keyword.get("target", "")
                modifier_values = [int(num) for num in re.findall(r"\d+", target)]

                # Update to the new structure
                effect["keywords"] = [
                    {
                        "keyword_name": keyword["keyword"],
                        "keyword_modifier": modifier_values
                    }
                ]
    return effect

In [ ]:
def update_link_effect(effect):
    """Update the structure for the 'Link' keyword by extracting the integer from the target."""
    if "keywords" in effect:
        for keyword in effect["keywords"]:
            if keyword.get("keyword") == "Link":
                # Extract the integer from the target string, e.g., "Costs 4 or More"
                target_value = keyword.get("target", "")
                match = re.search(r"\d+", target_value)
                if match:
                    # Update to the new structure with the integer as keyword_modifier
                    effect["keywords"] = [
                        {
                            "keyword_name": keyword["keyword"],
                            "keyword_modifier": int(match.group(0))  # Extracted integer
                        }
                    ]
    return effect

In [50]:
def reorder_keywords(effect):
    """Ensure 'keyword_name' appears before 'keyword_modifier' in each keyword."""
    if "keywords" in effect:
        for keyword in effect["keywords"]:
            # Reorder the dictionary
            if "keyword_name" in keyword and "keyword_modifier" in keyword:
                keyword_ordered = {
                    "keyword_name": keyword["keyword_name"],
                    "keyword_modifier": keyword["keyword_modifier"]
                }
                # Update the keyword with reordered keys
                keyword.clear()
                keyword.update(keyword_ordered)

    return effect

In [ ]:
# Walk through the folder and its subfolders
for root, _, files in os.walk(folder_path):
    for file in files:
        if file.endswith(".json"):
            file_path = os.path.join(root, file)

            # Open and read the JSON file
            with open(file_path, "r", encoding="utf-8") as json_file:
                try:
                    data = json.load(json_file)

                    # Check and process the "effects" field for the listed keywords
                    if "effects" in data:
                        for effect in data["effects"]:
                            update_link_effect(effect)

                    # Save the modified JSON back to the file
                    with open(file_path, "w", encoding="utf-8") as updated_file:
                        json.dump(data, updated_file, indent=4)
                    
                    print(f"Updated file: {file_path}")
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")


Updated file: ../json\BSS01\BSS01-001.json
Updated file: ../json\BSS01\BSS01-001_p1.json
Updated file: ../json\BSS01\BSS01-001_p2.json
Updated file: ../json\BSS01\BSS01-001_p3.json
Updated file: ../json\BSS01\BSS01-002.json
Updated file: ../json\BSS01\BSS01-003.json
Updated file: ../json\BSS01\BSS01-003_p1.json
Updated file: ../json\BSS01\BSS01-004.json
Updated file: ../json\BSS01\BSS01-004_p1.json
Updated file: ../json\BSS01\BSS01-005.json
Updated file: ../json\BSS01\BSS01-006.json
Updated file: ../json\BSS01\BSS01-007.json
Updated file: ../json\BSS01\BSS01-007_p1.json
Updated file: ../json\BSS01\BSS01-007_p2.json
Updated file: ../json\BSS01\BSS01-008.json
Updated file: ../json\BSS01\BSS01-009.json
Updated file: ../json\BSS01\BSS01-009_p1.json
Updated file: ../json\BSS01\BSS01-010.json
Updated file: ../json\BSS01\BSS01-010_p1.json
Updated file: ../json\BSS01\BSS01-010_p2.json
Updated file: ../json\BSS01\BSS01-011.json
Updated file: ../json\BSS01\BSS01-012.json
Updated file: ../json\BS